In [2]:
import uuid, time
from pyspark.sql.functions import col, lit
from src.config.parameters_config import get_notebook_parameters
from src.config.bronze_config import BronzeConfig
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.reader import DataFrameReader
from src.io.writer import DataFrameWriter

In [ ]:
layer = "bronze"

#Load Spark Session
spark = get_spark_session()
dataset = get_notebook_parameters()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

#Load yaml
bronze_cfg = BronzeConfig(dataset, layer)
bronze_cfg.load_yaml()

In [4]:
# Create variables from yaml
file_format = bronze_cfg.file_format
spark_options = bronze_cfg.spark_options
target_table = bronze_cfg.target_table
target_schema = bronze_cfg.target_schema
target_catalog = bronze_cfg.target_catalog

# Create logical variables
run_id = str(uuid.uuid4())
ingest_ts = spark.sql("select current_timestamp() as ts").collect()[0]["ts"]

# Create paths
file_path = f"/Volumes/{target_catalog}/landing/files/{dataset}/"
checkpoint_location_path = f"/Volumes/{target_catalog}/landing/checkpoints/bronze/{dataset}/"
schema_location_path = f"/Volumes/{target_catalog}/landing/schema/{dataset}/"

In [5]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")
log.info(f"[CTX] target_table={target_table} target_schema={target_schema} target_catalog={target_catalog}")
log.info(f"[CTX] file_path={file_path}")
log.info(f"[CTX] checkpoint_path={checkpoint_location_path}")
log.info(f"[CTX] schema_location_path={schema_location_path}")
log.info(f"[CTX] ingest_ts={ingest_ts}")

start = time.time()
log.info(f"[BRONZE][START] run_id={run_id}")

#Bronze Layer: Read File → Add Ingest Metadata → Add Processing Metadata → Write Delta

try:
    reader = DataFrameReader(spark)
    writer = DataFrameWriter(spark)

    # Read file
    df_raw = reader.read_file_stream(file_path, file_format, spark_options, schema_location_path)
    log.info(f"[STEP 1] READ: File ingested")
    
    # Add metadata columns
    df_ingest = bronze_cfg.add_metadata(df_raw, ingest_ts)
    log.info(f"[STEP 2] ADD_METADATA: Added _ingest_ts and source metadata columns")
    
    # Write table
    df_writed = writer.write_delta_stream(df_ingest, target_catalog, target_schema, target_table, checkpoint_location_path)
    log.info(f"[STEP 3] WRITE: Delta table")
        
    duration_s = round(time.time() - start, 3)
    log.info(f"[BRONZE][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[BRONZE][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise